In [1]:
translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']
joint_names = [
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip",
    "index_mcp_roll",
    "index_mcp_pitch",
    "index_pip",
    "index_dip",
    "middle_mcp_roll",
    "middle_mcp_pitch",
    "middle_pip",
    "middle_dip",
    "ring_mcp_pitch",
    "ring_pip",
    "ring_dip",
    "pinky_mcp_pitch",
    "pinky_pip",
    "pinky_dip"
]

thumb = {
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip"
}


In [2]:
import numpy as np
from collections import defaultdict
import scipy.spatial.transform as transform
import numpy as np
from scipy.spatial.transform import Rotation as R

grasp_poses =np.load('/home/guizhewei/guizhewei/bodex_dataset/grasp_poses_retargeted_bodex_bottle_1obj_good.npy', allow_pickle=True).item()
# grasp_poses =np.load('/home/ubuntu/Documents/DexYCB/grasp_poses_opt.npy', allow_pickle=True).item()
grasp_poses.keys()
# obj_idx = 3


dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219,

In [3]:
scene_scale_map = {k: v['scale_name'] for k, v in grasp_poses.items()}
scene_scale_map

{0: 'scale006',
 1: 'scale006',
 2: 'scale006',
 3: 'scale006',
 4: 'scale006',
 5: 'scale006',
 6: 'scale006',
 7: 'scale006',
 8: 'scale006',
 9: 'scale006',
 10: 'scale006',
 11: 'scale006',
 12: 'scale006',
 13: 'scale006',
 14: 'scale006',
 15: 'scale006',
 16: 'scale006',
 17: 'scale006',
 18: 'scale006',
 19: 'scale006',
 20: 'scale006',
 21: 'scale006',
 22: 'scale006',
 23: 'scale006',
 24: 'scale006',
 25: 'scale006',
 26: 'scale006',
 27: 'scale006',
 28: 'scale006',
 29: 'scale006',
 30: 'scale006',
 31: 'scale006',
 32: 'scale006',
 33: 'scale006',
 34: 'scale006',
 35: 'scale006',
 36: 'scale006',
 37: 'scale006',
 38: 'scale006',
 39: 'scale006',
 40: 'scale006',
 41: 'scale006',
 42: 'scale006',
 43: 'scale006',
 44: 'scale006',
 45: 'scale006',
 46: 'scale006',
 47: 'scale006',
 48: 'scale006',
 49: 'scale006',
 50: 'scale006',
 51: 'scale006',
 52: 'scale006',
 53: 'scale006',
 54: 'scale006',
 55: 'scale006',
 56: 'scale006',
 57: 'scale006',
 58: 'scale006',
 59: 's

In [4]:
grasp_poses[0]

{'original_idx': 0,
 'target_object_name': 'core_bottle_1071fa4cddb2da2fc8724d5673a063a6',
 'grasp_type': 'force_closure',
 'scale_name': 'scale006',
 'grasp_idx': 0,
 'scene_scale': 1.0,
 'obj_scale': array([0.06, 0.06, 0.06]),
 'target_pose_world': [Pose([0, 0, 0], [1, 0, 0, 0])],
 'shadow_qpos': array([-0.13348357, -0.05431002, -0.09412572,  0.8259839 , -0.06901718,
         0.29734516,  0.47389144, -0.19145474,  0.3490172 ,  0.49145466,
         0.49145466, -0.24863735,  0.63764757,  0.21769889,  0.21769889,
         0.26288855,  0.4582895 ,  0.7222829 ,  0.7222829 ,  0.50306815,
        -0.26941693,  0.60375863,  0.15717188,  0.15717188,  0.48667994,
         1.0803841 , -0.14304759, -0.03920776,  0.06292731], dtype=float32),
 'robot_pose': [array([-1.47739992e-01, -4.57922071e-02, -8.16908106e-02, -3.93920004e-01,
          5.85232079e-01, -8.88005197e-02,  1.59185976e-01,  7.06228316e-02,
         -4.71237319e-04,  5.83750725e-01,  5.85729241e-01, -7.23758578e-01,
          4.62

In [5]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def euler_to_rotation_matrix(euler_angles):
    rotation = R.from_euler('XYZ', euler_angles, degrees=False)
    return rotation.as_matrix()


def quaternion_to_rotation_matrix(quaternion):
    rotation = R.from_quat(quaternion)
    return rotation.as_matrix()


def object_pose_to_matrix(position, quaternion):
    """
    Converts object pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    quaternion = np.concatenate([quaternion[1:4], quaternion[0:1]]) # wxyz-> xyzw
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix


def hand_pose_to_matrix(position, quaternion):
    """
    Converts hand pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix


In [6]:
def get_world_frame_pose_for_opt(grasp_poses, obj_idx):
    ''' Obj pose '''
    obj_pos = grasp_poses[obj_idx]['target_pose_world'][0].p
    obj_quat = grasp_poses[obj_idx]['target_pose_world'][0].q
    object_pose = object_pose_to_matrix(obj_pos, obj_quat)
    # print(f"Object Position: {obj_pos}, Object Quaternion: {obj_quat}")
    # print(f"Object Pose: {object_pose}")

    ''' Hand pose in world frame (no transformation) '''
    hand_pos = grasp_poses[obj_idx]['robot_pose'][0][:3]
    hand_euler = grasp_poses[obj_idx]['robot_pose'][0][3:6]
    hand_quat = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_quat()
    hand_6drot = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_matrix()
    hand_6drot = hand_6drot[:, :2].T.ravel().tolist()
    # print(f"Hand Position (World Frame): {hand_pos}, Hand Quaternion: {hand_quat}")
    # print(f"Hand 6drot: {hand_6drot}")
    # print(grasp_poses[obj_idx]['robot_pose'][0])

    ''' Keep hand pose in world frame - no transformation '''
    print(f"World Frame Hand Euler (XYZ, rad): {hand_euler}")
    print(f"World Frame Hand Position: {hand_pos}")

    return hand_pos, hand_euler, object_pose, hand_6drot


In [7]:
ycb_optimize_dataset_list = []

for obj_idx in grasp_poses.keys():
    hand_pos, hand_euler, object_pose, hand_6drot = get_world_frame_pose_for_opt(grasp_poses, obj_idx)
    robot_pose_joint = grasp_poses[obj_idx]['robot_pose'][0]
    map_idx = [0, 5, 10, 15, 18, 1, 6, 11, 16, 2, 7, 12, 17, 3, 8, 13, 4, 9, 14]
    mapped_joint = [robot_pose_joint[6:][i] for i in map_idx]
    # print("mapped_joint", mapped_joint)
    robot_joint_dict = defaultdict(float)
    for i, joint_name in enumerate(joint_names):
        robot_joint_dict[joint_name] = mapped_joint[i]
    for i, name in enumerate(translation_names):
        robot_joint_dict[name] = hand_pos[i]
    for i, name in enumerate(rot_names):
        robot_joint_dict[name] = hand_euler[i]

    ycb_optimize_dataset = defaultdict(dict)
    ycb_optimize_dataset['qpos'] = robot_joint_dict
    print("obj_idx", obj_idx)
    print("ycb_optimize_dataset['qpos']", ycb_optimize_dataset['qpos'])
    ycb_optimize_dataset['object_code'] = grasp_poses[obj_idx]['target_object_name']
    ycb_optimize_dataset['object_pose'] = object_pose
    ycb_optimize_dataset['hand_rot6d'] = hand_6drot
    ycb_optimize_dataset['idx'] = obj_idx
    
    # 加载 contact map 相关数据 (来自 store_dexonomy_retarget.py)
    if 'contact_map_object' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['contact_map_object'] = grasp_poses[obj_idx]['contact_map_object']
    if 'contact_map_object_parts' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['contact_map_object_parts'] = grasp_poses[obj_idx]['contact_map_object_parts']
    if 'contact_map_object_part_names' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['contact_map_object_part_names'] = grasp_poses[obj_idx]['contact_map_object_part_names']
    if 'hand_surface_points' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['hand_surface_points'] = grasp_poses[obj_idx]['hand_surface_points']
    if 'hand_surface_normals' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['hand_surface_normals'] = grasp_poses[obj_idx]['hand_surface_normals']
    if 'object_point_cloud' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['object_point_cloud'] = grasp_poses[obj_idx]['object_point_cloud']
    if 'object_normal_cloud' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['object_normal_cloud'] = grasp_poses[obj_idx]['object_normal_cloud']
    if 'shadow_qpos' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['shadow_qpos'] = grasp_poses[obj_idx]['shadow_qpos']
    # 加载 scale 信息
    if 'scene_scale' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['scene_scale'] = grasp_poses[obj_idx]['scene_scale']
    if 'obj_scale' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['obj_scale'] = grasp_poses[obj_idx]['obj_scale']

    ycb_optimize_dataset_list.append(ycb_optimize_dataset)

print(f"\n加载了 {len(ycb_optimize_dataset_list)} 个样本 (World Frame)")
# 检查第一个样本是否包含 contact map 数据
if len(ycb_optimize_dataset_list) > 0:
    first_sample = ycb_optimize_dataset_list[0]
    print(f"第一个样本包含的键: {list(first_sample.keys())}")
    if 'contact_map_object' in first_sample:
        print(f"contact_map_object shape: {first_sample['contact_map_object'].shape}")
    if 'contact_map_object_parts' in first_sample:
        print(f"contact_map_object_parts shape: {first_sample['contact_map_object_parts'].shape}")
    if 'contact_map_object_part_names' in first_sample:
        print(f"contact_map_object_part_names: {first_sample['contact_map_object_part_names']}")
    if 'object_point_cloud' in first_sample:
        print(f"object_point_cloud shape: {first_sample['object_point_cloud'].shape}")
    if 'scene_scale' in first_sample:
        print(f"scene_scale: {first_sample['scene_scale']}")
    if 'obj_scale' in first_sample:
        print(f"obj_scale: {first_sample['obj_scale']}")
    if 'shadow_qpos' in first_sample:
        print(f"shadow_qpos: {first_sample['shadow_qpos']}")

ycb_optimize_dataset_list


World Frame Hand Euler (XYZ, rad): [-0.39392     0.58523208 -0.08880052]
World Frame Hand Position: [-0.14773999 -0.04579221 -0.08169081]
obj_idx 0
ycb_optimize_dataset['qpos'] defaultdict(<class 'float'>, {'thumb_cmc_roll': 0.1591859757900238, 'thumb_cmc_yaw': -0.7237585783004761, 'thumb_cmc_pitch': -0.00010444082727190107, 'thumb_mcp': -0.8322901129722595, 'thumb_ip': -0.6657488613665105, 'index_mcp_roll': 0.07062283158302307, 'index_mcp_pitch': 0.46291258931159973, 'index_pip': 0.45094314217567444, 'index_dip': 0.5033427352964879, 'middle_mcp_roll': -0.0004712373192887753, 'middle_mcp_pitch': 0.3344005346298218, 'middle_pip': 0.6116091012954712, 'middle_dip': 0.6981517891287803, 'ring_mcp_pitch': 0.5837507247924805, 'ring_pip': 0.5117158853530884, 'ring_dip': 0.5117158853530884, 'pinky_mcp_pitch': 0.5857292413711548, 'pinky_pip': 0.5678644995093346, 'pinky_dip': 0.5678644995093346, 'WRJTx': -0.14773999154567719, 'WRJTy': -0.04579220712184906, 'WRJTz': -0.08169081062078476, 'WRJRx': 

[defaultdict(dict,
             {'qpos': defaultdict(float,
                          {'thumb_cmc_roll': 0.1591859757900238,
                           'thumb_cmc_yaw': -0.7237585783004761,
                           'thumb_cmc_pitch': -0.00010444082727190107,
                           'thumb_mcp': -0.8322901129722595,
                           'thumb_ip': -0.6657488613665105,
                           'index_mcp_roll': 0.07062283158302307,
                           'index_mcp_pitch': 0.46291258931159973,
                           'index_pip': 0.45094314217567444,
                           'index_dip': 0.5033427352964879,
                           'middle_mcp_roll': -0.0004712373192887753,
                           'middle_mcp_pitch': 0.3344005346298218,
                           'middle_pip': 0.6116091012954712,
                           'middle_dip': 0.6981517891287803,
                           'ring_mcp_pitch': 0.5837507247924805,
                           'ring_pip': 0

In [8]:
# store dict in a specified path as npy file
import os
import json
output_path = '/home/guizhewei/guizhewei/grasp_pose_dataset/unoptimized/bodex/bodex_1obj_bottle_good.npy'
if not os.path.exists(os.path.dirname(output_path)):
    os.makedirs(os.path.dirname(output_path))
np.save(output_path, ycb_optimize_dataset_list, allow_pickle=True)
print(f"Saved to {output_path}")


Saved to /home/guizhewei/guizhewei/grasp_pose_dataset/unoptimized/bodex/bodex_1obj_bottle_good.npy
